# RAG Pipeline — LangGraph + Google Gemini (FREE) + FAISS

**100% Free Setup:**
- LLM: Google Gemini 2.5 Flash — free tier, no credit card needed
- Embeddings: `sentence-transformers` — runs locally, zero API cost
- Vector Store: FAISS — fully local
- Token Mapper Agent: tracks real-time token usage per node

> Get a FREE Gemini API key (30 seconds, no credit card): https://aistudio.google.com/apikey

---

## Cell 0 — Install Dependencies

Run this cell **once** to install all packages into the active Jupyter kernel's Python.
This ensures packages are installed into whichever Python version Jupyter is using.

In [2]:
import sys
import subprocess

print(f"Installing into: {sys.executable}")

packages = [
    "python-dotenv",
    "sentence-transformers",
    "faiss-cpu",
    "pypdf",
    "google-genai",
    "langgraph",
    "langchain",
    "langchain-core",
    "langchain-community",
    "pandas",
    "numpy",
    "tabulate",
    "tqdm",
]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install"] + packages + ["-q"],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("All packages installed successfully!")
else:
    print(result.stdout[-2000:] if result.stdout else "")
    print(result.stderr[-2000:] if result.stderr else "")

## Cell 1 — Imports & Setup

In [ ]:
import os
import sys
import time
import warnings
from pathlib import Path
from typing import TypedDict, List, Any
from dataclasses import dataclass

warnings.filterwarnings("ignore")

# Load API key from .env file
from dotenv import load_dotenv
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
if not GEMINI_API_KEY or GEMINI_API_KEY == "your_gemini_api_key_here":
    raise EnvironmentError(
        "GEMINI_API_KEY not set!\n"
        "  1. Go to: https://aistudio.google.com/apikey\n"
        "  2. Click Create API Key (free, no credit card)\n"
        "  3. Add to .env:  GEMINI_API_KEY=AIza..."
    )
print(f"Gemini API key loaded (...{GEMINI_API_KEY[-6:]})")

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from google import genai
from google.genai import types as genai_types
from langgraph.graph import StateGraph, END
import pandas as pd
from IPython.display import display, Markdown, HTML

print(f"Python: {sys.version}")
print("All imports successful!")

Gemini API key loaded (...MBWySU)
Python: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
All imports successful!


## Cell 2 — Configuration

> Set `PDF_PATH` and `GEMINI_MODEL` before running.

In [ ]:


# Name of your PDF file (must be in the same folder as this notebook)
PDF_PATH = "GRE_score.pdf"

GEMINI_MODEL = "gemini-2.5-flash"

# Text chunking settings
CHUNK_SIZE = 500     # characters per chunk
CHUNK_OVERLAP = 50   # overlap between adjacent chunks

# How many chunks to retrieve per question
TOP_K = 4

# Local embedding model — runs offline, no API key needed
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# ─────────────────────────────────────────────────────────────
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print(f"PDF Path:         {PDF_PATH}")
print(f"Gemini Model:     {GEMINI_MODEL}  [FREE TIER]")
print(f"Chunk Size:       {CHUNK_SIZE} chars (overlap: {CHUNK_OVERLAP})")
print(f"Top-K Retrieval:  {TOP_K} chunks")
print(f"Embedding Model:  {EMBEDDING_MODEL} (local / offline)")

PDF Path:         GRE_score.pdf
Gemini Model:     gemini-2.5-flash  [FREE TIER]
Chunk Size:       500 chars (overlap: 50)
Top-K Retrieval:  4 chunks
Embedding Model:  all-MiniLM-L6-v2 (local / offline)


## Cell 3 — Token Mapper Agent (Agent 2)

Wraps every LangGraph node with `start_node()` / `end_node()` hooks to track:
- Input tokens, output tokens, total tokens per node
- Elapsed time per node
- Renders a styled summary table at the end



In [ ]:
@dataclass
class NodeTokenRecord:
    """Stores token usage and timing for one LangGraph node."""
    node_name: str
    input_tokens: int = 0
    output_tokens: int = 0
    elapsed_ms: float = 0.0

    @property
    def total_tokens(self):
        return self.input_tokens + self.output_tokens


class TokenMapperAgent:
    """
    Agent 2 — Real-Time Token Mapper.

    Call start_node() before each node runs and end_node() after.
    Token counts come directly from the Gemini API response.
    Call report() to render the per-node styled summary table.
    """

    def __init__(self):
        self.records: List[NodeTokenRecord] = []
        self._timers = {}

    def start_node(self, name: str):
        self._timers[name] = time.perf_counter()
        print(f"  --> [{name}] starting...")

    def end_node(self, name: str, input_tokens: int = 0, output_tokens: int = 0):
        elapsed = 0.0
        if name in self._timers:
            elapsed = (time.perf_counter() - self._timers.pop(name)) * 1000
        record = NodeTokenRecord(
            node_name=name,
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            elapsed_ms=elapsed,
        )
        self.records.append(record)
        has_tokens = (input_tokens + output_tokens) > 0
        tok_str = f"{input_tokens} in / {output_tokens} out" if has_tokens else "(no LLM call)"
        print(f"  <-- [{name}] done  |  tokens: {tok_str}  |  {elapsed:.0f} ms")

    def report(self):
        """Render a styled per-node token usage table."""
        if not self.records:
            print("No records yet.")
            return

        rows = []
        total_in = 0
        total_out = 0
        total_ms = 0.0

        for r in self.records:
            total_in += r.input_tokens
            total_out += r.output_tokens
            total_ms += r.elapsed_ms
            rows.append({
                "Node": r.node_name,
                "Input Tokens": r.input_tokens if r.input_tokens else "—",
                "Output Tokens": r.output_tokens if r.output_tokens else "—",
                "Total Tokens": r.total_tokens if r.total_tokens else "—",
                "Time (ms)": f"{r.elapsed_ms:.0f}",
                "Cost": "$0.00 [FREE]",
            })

        rows.append({
            "Node": "TOTAL",
            "Input Tokens": total_in,
            "Output Tokens": total_out,
            "Total Tokens": total_in + total_out,
            "Time (ms)": f"{total_ms:.0f}",
            "Cost": "$0.00 [FREE TIER]",
        })

        df = pd.DataFrame(rows)
        display(Markdown("### Token Mapper Agent — Per-Node Usage"))
        display(HTML(
            df.to_html(index=False, border=0)
            .replace(
                "<table",
                "<table style='border-collapse:collapse;font-family:monospace;width:100%;'"
            )
            .replace(
                "<th>",
                "<th style='background:#1e3a5f;color:#e2e8f0;padding:8px 14px;"
                "border:1px solid #2d4a7a;text-align:left;'>"
            )
            .replace(
                "<td>",
                "<td style='padding:7px 14px;border:1px solid #2d4a7a;'>"
            )
        ))

    def reset(self):
        """Clear all records for a fresh run."""
        self.records.clear()
        self._timers.clear()


token_agent = TokenMapperAgent()
print("Token Mapper Agent ready.")

Token Mapper Agent ready.


## Cell 4 — Pipeline State & Helper

Defines the shared state that flows between all LangGraph nodes, and the text chunking helper.

In [ ]:
class PipelineState(TypedDict, total=False):
    """Shared state that flows through the entire LangGraph pipeline."""
    pdf_path: str
    question: str
    chunks: List[str]
    chunk_meta: List[dict]
    embeddings: Any
    faiss_index: Any
    retrieved: List[str]
    retrieved_meta: List[dict]
    answer: str
    token_usage: dict


def chunk_text(text: str, size: int, overlap: int) -> List[str]:
    """Split text into overlapping fixed-size character chunks."""
    chunks = []
    start = 0
    while start < len(text):
        chunk = text[start: start + size]
        chunks.append(chunk)
        start += size - overlap
    return [c.strip() for c in chunks if c.strip()]


print("PipelineState schema and chunk_text helper defined.")

PipelineState schema and chunk_text helper defined.


## Cell 5 — Load Embedding Model

In [ ]:
# Loads locally — no API key, no cost, runs offline
print(f"Loading embedding model '{EMBEDDING_MODEL}'...")
embedder = SentenceTransformer(EMBEDDING_MODEL)
print("Embedding model ready. (local, free, offline)")

Loading embedding model 'all-MiniLM-L6-v2'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4625.81it/s]


Embedding model ready. (local, free, offline)


## Cell 6 — RAG Pipeline Node Functions (Agent 1)

Each function is one node in the LangGraph pipeline.

In [ ]:
# ── NODE 1: ingest pdf ────────────────────────────────────────
def ingest_pdf(state: PipelineState) -> PipelineState:
    """Parse the PDF page by page and split text into overlapping chunks."""
    token_agent.start_node("ingest_pdf")

    pdf_path = state["pdf_path"]
    if not Path(pdf_path).exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    reader = PdfReader(pdf_path)
    chunks = []
    meta = []

    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        page_chunks = chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)
        for i, ch in enumerate(page_chunks):
            chunks.append(ch)
            meta.append({"page": page_num, "chunk_index": i})

    token_agent.end_node("ingest_pdf")
    print(f"     {len(reader.pages)} pages -> {len(chunks)} chunks")
    return {**state, "chunks": chunks, "chunk_meta": meta}


# ── NODE 2: embed_and_index ───────────────────────────────────
def embed_and_index(state: PipelineState) -> PipelineState:
    """Embed all chunks locally using sentence-transformers and build a FAISS index."""
    token_agent.start_node("embed_and_index")

    chunks = state["chunks"]
    print(f"     Embedding {len(chunks)} chunks locally (free, no API)...")

    emb = embedder.encode(
        chunks,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    # IndexFlatIP = cosine similarity (since embeddings are normalized)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)

    token_agent.end_node("embed_and_index")
    print(f"     FAISS index built: {index.ntotal} vectors, dim={emb.shape[1]}")
    return {**state, "embeddings": emb, "faiss_index": index}


# ── NODE 3: retrieve ──────────────────────────────────────────
def retrieve(state: PipelineState) -> PipelineState:
    """Embed the question and retrieve top-K most similar chunks via FAISS."""
    token_agent.start_node("retrieve")

    q_emb = embedder.encode(
        [state["question"]],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    scores, indices = state["faiss_index"].search(q_emb, TOP_K)
    retrieved = [state["chunks"][i] for i in indices[0] if i >= 0]
    retrieved_meta = [state["chunk_meta"][i] for i in indices[0] if i >= 0]

    score_str = ", ".join(f"{s:.3f}" for s in scores[0])
    print(f"     Top-{TOP_K} similarity scores: [{score_str}]")
    for m in retrieved_meta:
        print(f"       -> Page {m['page']}, chunk #{m['chunk_index']}")

    token_agent.end_node("retrieve")
    return {**state, "retrieved": retrieved, "retrieved_meta": retrieved_meta}


# ── NODE 4: generate_answer ───────────────────────────────────
def generate_answer(state: PipelineState) -> PipelineState:
    """Build the RAG prompt from retrieved chunks and call Gemini (free tier)."""
    token_agent.start_node("generate_answer")

    context_parts = []
    for i, (chunk, m) in enumerate(zip(state["retrieved"], state["retrieved_meta"])):
        context_parts.append(f"[Chunk {i + 1} | Page {m['page']}]\n{chunk}")
    context = "\n\n---\n\n".join(context_parts)

    prompt = (
        "You are a precise document assistant. "
        "Answer ONLY using the context below. "
        "If the answer is not present, say: This information is not in the document. "
        "Cite the chunk number(s) you used.\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION: {state['question']}\n\n"
        "ANSWER:"
    )

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=genai_types.GenerateContentConfig(
            temperature=0.2,
            max_output_tokens=1024,
        ),
    )

    answer = response.text.strip()
    usage = response.usage_metadata
    token_usage = {
        "input_tokens": usage.prompt_token_count or 0,
        "output_tokens": usage.candidates_token_count or 0,
        "total_tokens": usage.total_token_count or 0,
    }

    token_agent.end_node(
        "generate_answer",
        input_tokens=token_usage["input_tokens"],
        output_tokens=token_usage["output_tokens"],
    )
    return {**state, "answer": answer, "token_usage": token_usage}


# ── NODE 5: token_summary (Token Mapper Agent node) ───────────
def token_summary(state: PipelineState) -> PipelineState:
    """Signal node — the Token Mapper report is printed in Cell 8."""
    token_agent.start_node("token_summary")
    token_agent.end_node("token_summary")
    return state


print("All 5 RAG pipeline nodes defined.")

All 5 RAG pipeline nodes defined.


## Cell 7 — Build & Compile the LangGraph

In [19]:
builder = StateGraph(PipelineState)

# Agent 1 — RAG nodes
builder.add_node("ingest_pdf", ingest_pdf)
builder.add_node("embed_and_index", embed_and_index)
builder.add_node("retrieve", retrieve)
builder.add_node("generate_answer", generate_answer)

# Agent 2 — Token Mapper node
builder.add_node("token_summary", token_summary)

# Wire the pipeline (linear flow)
builder.set_entry_point("ingest_pdf")
builder.add_edge("ingest_pdf", "embed_and_index")
builder.add_edge("embed_and_index", "retrieve")
builder.add_edge("retrieve", "generate_answer")
builder.add_edge("generate_answer", "token_summary")
builder.add_edge("token_summary", END)

pipeline = builder.compile()

print("LangGraph pipeline compiled successfully!")
print()
print("  ingest_pdf")
print("      |")
print("  embed_and_index   <- local sentence-transformers + FAISS")
print("      |")
print("  retrieve          <- semantic search, local")
print("      |")
print("  generate_answer   <- Gemini 2.5 Flash (FREE tier)")
print("      |")
print("  token_summary     <- Token Mapper Agent")
print("      |")
print("    END")

LangGraph pipeline compiled successfully!

  ingest_pdf
      |
  embed_and_index   <- local sentence-transformers + FAISS
      |
  retrieve          <- semantic search, local
      |
  generate_answer   <- Gemini 2.5 Flash (FREE tier)
      |
  token_summary     <- Token Mapper Agent
      |
    END


## Cell 8 — Run the Pipeline


In [33]:
#  Edit your question here
QUESTION = "What is this document about?"

token_agent.reset()

print("=" * 60)
print(f"  PDF:      {PDF_PATH}")
print(f"  Question: {QUESTION}")
print(f"  Model:    {GEMINI_MODEL}")
print("=" * 60)

final_state = pipeline.invoke({
    "pdf_path": PDF_PATH,
    "question": QUESTION,
})

# Print the answer only
print()
print("=" * 60)
print("  ANSWER")
print("=" * 60)
display(Markdown(final_state["answer"]))

  PDF:      GRE_score.pdf
  Question: What is this document about?
  Model:    gemini-2.5-flash
  --> [ingest_pdf] starting...
  <-- [ingest_pdf] done  |  tokens: (no LLM call)  |  78 ms
     3 pages -> 15 chunks
  --> [embed_and_index] starting...
     Embedding 15 chunks locally (free, no API)...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.95it/s]


  <-- [embed_and_index] done  |  tokens: (no LLM call)  |  260 ms
     FAISS index built: 15 vectors, dim=384
  --> [retrieve] starting...
     Top-4 similarity scores: [0.200, 0.155, 0.147, 0.123]
       -> Page 1, chunk #1
       -> Page 3, chunk #0
       -> Page 2, chunk #8
       -> Page 1, chunk #0
  <-- [retrieve] done  |  tokens: (no LLM call)  |  10 ms
  --> [generate_answer] starting...
  <-- [generate_answer] done  |  tokens: 669 in / 17 out  |  6803 ms
  --> [token_summary] starting...
  <-- [token_summary] done  |  tokens: (no LLM call)  |  0 ms

  ANSWER


This document is a TEST TAKER SCORE REPORT.
Chunks 2, 4

## Cell 9 — Token Mapper Agent Report

Per-node token usage across the full pipeline. Gemini free tier — **cost: $0.00**.

In [34]:
token_agent.report()

if "token_usage" in final_state:
    u = final_state["token_usage"]
    print()
    print("Raw Gemini token counts:")
    print(f"  Prompt tokens:    {u['input_tokens']:,}")
    print(f"  Candidate tokens: {u['output_tokens']:,}")
    print(f"  Total tokens:     {u['total_tokens']:,}")
    print()
    print("Gemini 2.5 Flash free tier limits:")
    print("  500 requests/day | 1,000,000 tokens/day | $0.00")

### Token Mapper Agent — Per-Node Usage

Node,Input Tokens,Output Tokens,Total Tokens,Time (ms),Cost
ingest_pdf,—,—,—,78,$0.00 [FREE]
embed_and_index,—,—,—,260,$0.00 [FREE]
retrieve,—,—,—,10,$0.00 [FREE]
generate_answer,669,17,686,6803,$0.00 [FREE]
token_summary,—,—,—,0,$0.00 [FREE]
TOTAL,669,17,686,7151,$0.00 [FREE TIER]



Raw Gemini token counts:
  Prompt tokens:    669
  Candidate tokens: 17
  Total tokens:     1,668

Gemini 2.5 Flash free tier limits:
  500 requests/day | 1,000,000 tokens/day | $0.00


## Cell 10 — Follow-up Questions (no re-embedding)

Reuses the already-built FAISS index. Only `retrieve` and `generate_answer` re-run — no re-ingestion or re-embedding needed.

In [36]:
def ask(question: str, existing_state: PipelineState):
    """
    Ask a follow-up question without re-ingesting or re-embedding the PDF.
    Reuses the FAISS index from the previous full run.
    """
    mini_builder = StateGraph(PipelineState)
    mini_builder.add_node("retrieve", retrieve)
    mini_builder.add_node("generate_answer", generate_answer)
    mini_builder.add_node("token_summary", token_summary)
    mini_builder.set_entry_point("retrieve")
    mini_builder.add_edge("retrieve", "generate_answer")
    mini_builder.add_edge("generate_answer", "token_summary")
    mini_builder.add_edge("token_summary", END)
    mini_pipeline = mini_builder.compile()

    token_agent.reset()
    print(f"Question: {question}\n")
    result = mini_pipeline.invoke({**existing_state, "question": question})
    display(Markdown("**Answer:**\n\n" + result["answer"]))
    token_agent.report()
    return result


final_state = ask("What sections does this GRE test has?", final_state)


Question: What sections does this GRE test has?

  --> [retrieve] starting...
     Top-4 similarity scores: [0.561, 0.560, 0.557, 0.525]
       -> Page 2, chunk #6
       -> Page 3, chunk #1
       -> Page 2, chunk #5
       -> Page 2, chunk #2
  <-- [retrieve] done  |  tokens: (no LLM call)  |  21 ms
  --> [generate_answer] starting...
  <-- [generate_answer] done  |  tokens: 566 in / 38 out  |  2429 ms
  --> [token_summary] starting...
  <-- [token_summary] done  |  tokens: (no LLM call)  |  0 ms


**Answer:**

For detailed information about your performance on the Verbal Reasoning and Quantitative Reasoning sections of the computer-delivered GRE General Test, access the free GRE Diagnostic Service from your ETS account.
Chunk 3

### Token Mapper Agent — Per-Node Usage

Node,Input Tokens,Output Tokens,Total Tokens,Time (ms),Cost
retrieve,—,—,—,21,$0.00 [FREE]
generate_answer,566,38,604,2429,$0.00 [FREE]
token_summary,—,—,—,0,$0.00 [FREE]
TOTAL,566,38,604,2450,$0.00 [FREE TIER]
